In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import part1

from datasets import load_dataset
mnli = load_dataset("nyu-mll/multi_nli")

In [31]:

%%time
my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

# now for a subsample of the whole corpus since its too much like 400.000 examples or something. chose 5.000 for building. since otherwise im waiting for too long.

n = 50000
sample_train = mnli["train"].shuffle(seed=0).select(range(n))
print(sample_train)

# Put premise and Hypothesis together in one string for the tokeniser.
texts = [p + " " + h for p, h in zip(sample_train["premise"], sample_train["hypothesis"])]

for seed in my_seeds:  
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(texts, sample_train["label"])), 
            k=num_folds,
            seed=seed
        )
    )

Dataset({
    features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
    num_rows: 50000
})
CPU times: user 2.34 s, sys: 33.4 ms, total: 2.37 s
Wall time: 2.45 s


In [24]:
from collections import Counter
Counter(sample_train["label"])
# label distribution still 1/3 each so went well.

Counter({0: 16840, 2: 16691, 1: 16469})

In [4]:
# Now lets do the tokeniser.
# first take all the premise and hypothesis of the train set.

all_train_texts = [p + " " + h for p, h in zip(mnli["train"]["premise"], mnli["train"]["hypothesis"])]

# choose same vocab size as in tutorial; bc its smaller and so a bit faster.
vocab_size = 1000
# min freq of a token to merge for bpe.
min_frequency = 2
tokenizer = part1.experiment.train_tokenizer(all_train_texts,vocab_size,min_frequency)
results_folder = Path.cwd().parent / "results"
tokenizer.save(f"{results_folder}/BPE_tokenizer_vs_{vocab_size}_mf_{min_frequency}.json")

In [5]:
vocab = tokenizer.get_vocab()
for _, (t, i) in zip(range(10), vocab.items()):
    print(f"{t} -> {i}")

ru -> 557
▁ke -> 909
¡ -> 94
gr -> 563
▁from -> 329
▁them -> 429
en -> 172
▁they're -> 904
é -> 110
▁V -> 546


In [6]:
print(all_train_texts[0])
print(tokenizer.encode(all_train_texts[0]).tokens)

Conceptually cream skimming has two basic dimensions - product and geography. Product and geography are what make cream skimming work. 
['▁C', 'on', 'ce', 'pt', 'ually', '▁c', 're', 'am', '▁s', 'k', 'im', 'm', 'ing', '▁has', '▁two', '▁bas', 'ic', '▁d', 'im', 'ens', 'ions', '▁', '-', '▁produ', 'ct', '▁and', '▁g', 'e', 'og', 'ra', 'p', 'h', 'y.', '▁P', 'ro', 'du', 'ct', '▁and', '▁g', 'e', 'og', 'ra', 'p', 'h', 'y', '▁are', '▁what', '▁make', '▁c', 're', 'am', '▁s', 'k', 'im', 'm', 'ing', '▁work', '.', '▁']


In [7]:
token_indicator_encoder = part1.clslib.TokenIndicator(tokenizer)
print(len(token_indicator_encoder))

1000


In [8]:
print(mnli["train"].features["label"])

ClassLabel(names=['entailment', 'neutral', 'contradiction'])


In [ ]:
# prep outcome space!

from collections import OrderedDict

# than as in the tutorial the outcome space:
mnli_outcome_space = part1.clslib.NamedClasses(['entailment', 'neutral', 'contradiction'])
print(mnli_outcome_space)



0: entailment, 1: neutral, 2: contradiction


In [32]:
%%time
my_classifiers = OrderedDict()
# next the first model; loglin classifier as in the notebook.

my_classifiers['logistic_indicator-enc'] = []
for run, seed in enumerate(my_seeds):  
    run_folder = f"{results_folder}/logistic_indicator-enc/seed-{seed}"
    my_classifiers['logistic_indicator-enc'].append(
        part1.experiment.cross_validation_experiment(        
            data_splits=my_cross_validation_sets[run],
            outcome_space=mnli_outcome_space,
            encoder=token_indicator_encoder, 
            model_option='logistic',
            save_to=run_folder,
            loglin_args={
                'max_iter': 1000
            }
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

/Volumes/Extreme SSD/UNI/master AI/Natural Language Processing 1/Github Repo/NLP1-Natural_Language_Inference-Project/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Volumes/Extreme SSD/UNI/master AI/Natural Language Processing 1/Github Repo/NLP1-Natural_Language_Inference-Project/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Volumes/Extreme SSD/UNI/master AI/Natural Language Processing 1/Github Repo/NLP1-Natural_Language_Inference-Project/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in

CPU times: user 1min 34s, sys: 867 ms, total: 1min 34s
Wall time: 1min 36s


In [33]:
print(my_classifiers["logistic_indicator-enc"][0]["accuracies"])

OrderedDict({0: 0.4493, 1: 0.4398, 2: 0.4437, 3: 0.4466, 4: 0.447})


i tried with max iter of 100 and 1000. almost no change but the latter does converge so its fullty trained the former didnt. results indicate on guesswork since this is also about the ratio of te classes. just a little more.
tthen also tried n=50000 o n the data sample size since it might not have enough to train on. 5000 with max iter 1000 gave OrderedDict({0: 0.377, 1: 0.379, 2: 0.403, 3: 0.367, 4: 0.356}).  max iter 100 didnt do much different. then now for n=50000 with max iter 1000 i do see a bigger difference: OrderedDict({0: 0.4493, 1: 0.4398, 2: 0.4437, 3: 0.4466, 4: 0.447}) 
o it does seem like more data makes this a bit bteer. so lets do more i guess. but 10x more data didnt make it 10x better more like 1/5th or a bit less better
also the difference between fold performance is smaller so that means less noise which is better. ill try 100k too make its own result thingy for it.
OrderedDict({0: 0.4561, 1: 0.45895, 2: 0.45695, 3: 0.45645, 4: 0.4601}) is hte result for that. not rlly worth it with thatm uch mroe time for so little gain. os lets stick with 50k.

